# Coral Heat Stress Prediction

Predicts the onset of coral bleaching stress using XGBoost and Logistic Regression  
with temporal k-fold cross-validation across one or more reef sites.

**Author:** Marybeth Arcodia  

**Usage:** Set `SITES_TO_RUN` in the Configuration cell to run 1–3 sites.  
When new data files are available, update `DATA_FILENAME_TEMPLATE` and `DATE_CUTOFF`.

In [ ]:
import sys
import os
sys.path.insert(0, '../src')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import xgboost
import shap
import json
import warnings
from pathlib import Path
from sklearn.metrics import mean_absolute_error

import preprocess as preprocess_module
from data_split import create_labels_and_features, create_splits
import train_test_model

warnings.filterwarnings('ignore')
np.set_printoptions(threshold=np.inf)
pd.set_option('display.max_rows', None)

print(f'scikit-learn {sklearn.__version__} | xgboost {xgboost.__version__}')

## Configuration

Edit `SITES_TO_RUN` to choose which sites to process.  
Update `DATA_FILENAME_TEMPLATE` and `DATE_CUTOFF` when new data is added.

In [ ]:
# ── Sites to process ──────────────────────────────────────────────────────────
# Edit this list to run 1, 2, or all 3 sites.
SITES_TO_RUN = ['sombrero', 'molasses', 'sk']

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR   = Path('../Data')    # Place CSV files here
OUTPUT_DIR = Path('../output')  # All outputs written here

# ── Experiment label ──────────────────────────────────────────────────────────
EXP_NAME = 'exp402'

# ── Data file ─────────────────────────────────────────────────────────────────
# Update DATE_CUTOFF when new data files are added.
DATA_FILENAME_TEMPLATE = '{site}_preds.csv'
DATE_CUTOFF = 2114  # Row index cutoff — increase to include more data

# ── Prediction parameters ─────────────────────────────────────────────────────
STRESS_THRESHOLD = 4
PREDICTION_SPAN  = 6
VAL_NO_ONSET     = PREDICTION_SPAN + 1  # = 7 (label for 'no onset' class)
SUMMER_START     = 21
SUMMER_END       = 40
WEEKS_PER_YEAR   = SUMMER_END - SUMMER_START + 1  # = 20
N_FOLDS          = 5
TOLERANCE        = 1  # ±weeks for onset accuracy

# ── Features to use ───────────────────────────────────────────────────────────
# Edit this list to add/remove features. All must be present in the CSV.
COLUMNS_TO_KEEP = [
    'week',
    'meanDhw',
    'meanHs',
    'weeks_until_onset',  # label — always keep
    'meanDhw_lag_1',
    'meanDhw_lag_2',
    'meanSsta',
    'airTempInter',
    'zonalWindInter',
    'oniInter',
    'loop_curr_sst',
    'solar_flux',
]

# ── Sample weights ────────────────────────────────────────────────────────────
WEIGHT_NO_ONSET = 0.02  # Down-weight the majority 'no onset' class
WEIGHT_ONSET    = 1.0

# ── Per-site XGBoost hyperparameters (tuned via hyperparameter_sweep.ipynb) ───
SITE_XGB_PARAMS = {
    'sombrero': {'learning_rate': 0.1, 'reg_lambda': 0.0},
    'sk':       {'learning_rate': 0.5, 'reg_lambda': 0.0},
    'molasses': {'learning_rate': 0.3, 'reg_lambda': 5.0},
}

# ── Base XGBoost parameters (shared across sites) ─────────────────────────────
BASE_XGB_PARAMS = {
    'n_estimators':     20000,
    'max_depth':        10,
    'verbosity':        1,
    'objective':        'multi:softmax',
    'booster':          'gbtree',
    'tree_method':      'auto',
    'n_jobs':           1,
    'gamma':            0.0,
    'min_child_weight': 1,
    'max_delta_step':   0,
    'subsample':        0.7,
    'colsample_bytree': 1.0,
    'colsample_bylevel':1.0,
    'colsample_bynode': 1.0,
    'reg_alpha':        1.0,
    'random_state':     13,
}

# ── Display names for plots ────────────────────────────────────────────────────
SITE_DISPLAY_NAMES = {
    'sombrero': 'Sombrero',
    'sk':       'Sand Key',
    'molasses': 'Molasses',
}

print('Configuration loaded.')
print(f'Sites to run: {SITES_TO_RUN}')

## Main Processing Loop

For each site: preprocess → cross-validate (XGB + Linear) → compute metrics → save all outputs.

In [ ]:
all_site_results = {}  # Populated per site; used by visualization cells below

for reef_location in SITES_TO_RUN:
    site_name = SITE_DISPLAY_NAMES.get(reef_location, reef_location.capitalize())
    print(f"\n{'='*60}")
    print(f'Processing: {site_name}')
    print(f"{'='*60}")

    # ── Output directories ────────────────────────────────────────────────────
    site_out   = OUTPUT_DIR / EXP_NAME / reef_location
    models_dir = site_out / 'models'
    plots_dir  = site_out / 'plots'
    models_dir.mkdir(parents=True, exist_ok=True)
    plots_dir.mkdir(parents=True, exist_ok=True)

    # ── Preprocessing ─────────────────────────────────────────────────────────
    preprocessor = preprocess_module.CoralBleachingPreprocessor(
        str(DATA_DIR) + '/',
        reef_location,
        STRESS_THRESHOLD,
        VAL_NO_ONSET,
        PREDICTION_SPAN,
        WEEKS_PER_YEAR,
    )

    preprocessor.load_data(DATA_FILENAME_TEMPLATE.format(site=reef_location))
    preprocessor.preprocess_time_variables()
    preprocessor.filter_data(max_index=DATE_CUTOFF)
    preprocessor.calculate_weeks_until_onset(PREDICTION_SPAN)

    # Derive n_years from data before year column is dropped
    n_years = preprocessor.data['year'].nunique()
    print(f'Years in dataset: {n_years}')

    # Baseline predictions (uses year column — must be called before select_columns)
    baseline_preds = preprocessor.create_baseline_preds(
        SUMMER_START, n_years, PREDICTION_SPAN, VAL_NO_ONSET
    )
    most_frequent_onset_week = preprocessor.identify_most_frequent_onset_week(verbose=False)

    # Add lag features, then finalize columns and filter to summer
    preprocessor.add_lag_features({'meanDhw': [1, 2]})
    preprocessor.select_columns(COLUMNS_TO_KEEP)
    preprocessor.drop_missing_values()
    preprocessor.filter_summer_weeks(SUMMER_START, SUMMER_END)

    preprocessor.plot_mean_dhw()
    preprocessor.plot_onset_histogram()

    processed_data = preprocessor.data
    print(f'Processed data shape: {processed_data.shape}')

    # ── Labels, features, splits ──────────────────────────────────────────────
    labels, features = create_labels_and_features(processed_data)
    splits = create_splits(N_FOLDS, n_years, WEEKS_PER_YEAR)
    feature_names = list(processed_data.drop(columns=['weeks_until_onset']).columns)
    renamed_features = train_test_model.rename_features(feature_names)

    # ── Model parameters for this site ────────────────────────────────────────
    fd = {**BASE_XGB_PARAMS, **SITE_XGB_PARAMS.get(reef_location, {})}

    # ── Cross-validation ──────────────────────────────────────────────────────
    array_weeks = np.append(range(0, WEEKS_PER_YEAR), VAL_NO_ONSET)

    all_preds_xgb   = []
    all_preds_lin   = []
    all_fold_shap   = []
    all_fold_ev     = []  # expected values per fold
    mae_all_xgb     = np.zeros([WEEKS_PER_YEAR + 1, N_FOLDS])
    mae_all_lin     = np.zeros([WEEKS_PER_YEAR + 1, N_FOLDS])

    for fold, (train_idx, test_idx) in enumerate(splits):
        X_train, Y_train = features[train_idx], labels[train_idx]
        X_test,  Y_test  = features[test_idx],  labels[test_idx]

        print(f'\n  Fold {fold+1}: {len(train_idx)} train / {len(test_idx)} test')

        weights_train = np.where(
            Y_train == VAL_NO_ONSET, WEIGHT_NO_ONSET, WEIGHT_ONSET
        )

        # XGBoost
        xgb_model, y_pred_train_xgb, y_pred_xgb, _, mae_xgb = (
            train_test_model.train_test_xgboost(
                X_train, Y_train, X_test, Y_test, fd, weights_train, fold, WEEKS_PER_YEAR
            )
        )
        all_preds_xgb.append(np.round(y_pred_xgb, 0))
        mae_all_xgb[:, fold] = train_test_model.calculate_weekly_errors(
            Y_test, y_pred_xgb, array_weeks, WEEKS_PER_YEAR
        )

        # SHAP expected values from training logits
        logits_train = xgb_model.predict(X_train, output_margin=True)
        base_values  = np.average(logits_train, axis=0, weights=weights_train)
        explainer    = shap.TreeExplainer(xgb_model)
        explainer.expected_value = list(base_values)
        all_fold_ev.append(np.array(base_values))

        # Per-fold SHAP plots
        class_names = [str(i) for i in range(VAL_NO_ONSET + 1)]
        test_year_range = [
            int(test_idx[0]  / WEEKS_PER_YEAR + 1985),
            int(test_idx[-1] / WEEKS_PER_YEAR + 1985),
        ]
        train_test_model.save_shap_plot_multiclass(
            explainer, X_test, feature_names, fold, base_values,
            test_year_range, plots_dir, class_names,
        )

        shap_fold = explainer.shap_values(X_test, check_additivity=True)
        all_fold_shap.append(shap_fold)

        # Save XGBoost model
        xgb_model.save_model(
            str(models_dir / f'{reef_location}_xgb_fold_{fold+1}.model')
        )

        # Linear model
        lin_model, _, y_pred_lin, mae_lin = train_test_model.train_test_linear(
            X_train, Y_train, X_test, Y_test, weights_train, fold, WEEKS_PER_YEAR
        )
        all_preds_lin.append(np.round(y_pred_lin, 0))
        mae_all_lin[:, fold] = train_test_model.calculate_weekly_errors(
            Y_test, y_pred_lin, array_weeks, WEEKS_PER_YEAR
        )

        # Save Linear model coefficients
        np.save(
            str(models_dir / f'{reef_location}_linear_fold_{fold+1}.npy'), lin_model
        )

    # ── Aggregate predictions and apply onset mask ────────────────────────────
    all_preds_xgb_cat = np.concatenate(all_preds_xgb)
    all_preds_lin_cat = np.concatenate(all_preds_lin)

    masked_xgb, masked_lin, masked_base = train_test_model.apply_onset_mask(
        labels, all_preds_xgb_cat, all_preds_lin_cat, baseline_preds, WEEKS_PER_YEAR
    )

    # ── MAE per week ──────────────────────────────────────────────────────────
    mae_xgb_weekly  = np.nanmean(mae_all_xgb, axis=1)
    mae_lin_weekly  = np.nanmean(mae_all_lin, axis=1)
    mae_freq_weekly = np.zeros(WEEKS_PER_YEAR + 1)
    for count, week in enumerate(array_weeks):
        idx = np.where(labels == week)[0]
        mae_freq_weekly[count] = (
            mean_absolute_error(labels[idx], baseline_preds[idx]) if idx.size > 0 else np.nan
        )

    # ── Classification metrics ────────────────────────────────────────────────
    cutoff = np.repeat(VAL_NO_ONSET, len(labels))
    models_eval = {
        'XGB':       masked_xgb,
        'Linear':    masked_lin,
        'Frequency': masked_base,
    }
    site_metrics = {}
    metrics_for_plot = {}  # [onset_acc%, no_onset_acc%, fa_rate%, miss_rate%, f1]

    for model_name, preds in models_eval.items():
        hit  = (labels < VAL_NO_ONSET) & (np.abs(preds - labels) <= TOLERANCE)
        miss = (labels < VAL_NO_ONSET) & (np.abs(preds - labels) > TOLERANCE)
        fa   = (labels == VAL_NO_ONSET) & (preds < cutoff)
        tno  = (labels == VAL_NO_ONSET) & (preds >= cutoff)

        n_hit, n_miss = int(np.sum(hit)),  int(np.sum(miss))
        n_fa,  n_tno  = int(np.sum(fa)),   int(np.sum(tno))

        onset_acc   = n_hit / (n_hit + n_miss)  if (n_hit + n_miss) > 0 else 0
        no_onset    = n_tno / (n_tno + n_fa)    if (n_tno + n_fa)   > 0 else 0
        fa_rate     = n_fa  / (n_fa  + n_tno)   if (n_fa  + n_tno)  > 0 else 0
        miss_rate   = n_miss/ (n_hit + n_miss)  if (n_hit + n_miss) > 0 else 0
        precision   = n_hit / (n_hit + n_fa)    if (n_hit + n_fa)   > 0 else 0
        recall      = n_hit / (n_hit + n_miss)  if (n_hit + n_miss) > 0 else 0
        f1          = (2 * precision * recall / (precision + recall)
                       if (precision + recall) > 0 else 0)

        site_metrics[model_name] = {
            'onset_accuracy':    round(onset_acc * 100, 2),
            'no_onset_accuracy': round(no_onset  * 100, 2),
            'false_alarm_rate':  round(fa_rate   * 100, 2),
            'missed_onset_rate': round(miss_rate * 100, 2),
            'precision':         round(precision,       4),
            'recall':            round(recall,          4),
            'f1_score':          round(f1,              4),
            'counts': {
                'hit': n_hit, 'miss': n_miss,
                'false_alarm': n_fa, 'true_no_onset': n_tno,
            },
        }
        metrics_for_plot[model_name] = [
            onset_acc * 100, no_onset * 100,
            fa_rate   * 100, miss_rate * 100,
            f1,
        ]

        print(f'  {model_name}: onset_acc={onset_acc*100:.1f}%  f1={f1:.3f}')

    # ── Save metrics JSON ─────────────────────────────────────────────────────
    metrics_payload = {
        'site':         reef_location,
        'site_display': site_name,
        'exp_name':     EXP_NAME,
        'tolerance':    TOLERANCE,
        'n_years':      n_years,
        'models':       site_metrics,
    }
    with open(site_out / 'metrics.json', 'w') as f:
        json.dump(metrics_payload, f, indent=2)
    print(f'Metrics saved → {site_out / "metrics.json"}')

    # ── SHAP combined figure ──────────────────────────────────────────────────
    all_shap_stacked = np.vstack(all_fold_shap)  # (n_samples, n_features, n_classes)
    all_fold_ev_arr  = np.array(all_fold_ev)      # (n_folds, n_classes)
    X_values = processed_data.drop(columns=['weeks_until_onset']).values

    train_test_model.save_shap_summary_figure(
        all_shap_stacked,
        all_fold_ev_arr,
        X_values,
        renamed_features,
        VAL_NO_ONSET,
        plots_dir / f'{reef_location}_shap_summary_all_folds.png',
        site_display_name=site_name,
    )

    # ── Performance combined figure ───────────────────────────────────────────
    train_test_model.save_performance_figure(
        labels,
        masked_xgb, masked_lin, masked_base,
        mae_xgb_weekly, mae_lin_weekly, mae_freq_weekly,
        metrics_for_plot,
        VAL_NO_ONSET,
        TOLERANCE,
        plots_dir / f'{reef_location}_performance.png',
        site_display_name=site_name,
    )

    # ── Metrics table figure ──────────────────────────────────────────────────
    table_rows = []
    for mn, mv in site_metrics.items():
        c = mv['counts']
        table_rows.append([
            mn,
            f"{c['hit']}/{c['hit']+c['miss']}",
            f"{c['miss']}/{c['hit']+c['miss']}",
            f"{c['false_alarm']}/{c['false_alarm']+c['true_no_onset']}",
            f"{c['true_no_onset']}/{c['false_alarm']+c['true_no_onset']}",
            f"{mv['precision']:.3f}",
            f"{mv['recall']:.3f}",
            f"{mv['f1_score']:.3f}",
        ])
    df_metrics_table = pd.DataFrame(table_rows, columns=[
        'Model', 'Correct\nOnset', 'Missed\nOnset',
        'False\nAlarm', 'True\nNo Onset',
        'Precision', 'Recall', 'F1 Score',
    ])
    train_test_model.save_metrics_table(
        df_metrics_table,
        plots_dir / f'{reef_location}_metrics_table_{TOLERANCE}tol.png',
    )

    # ── Store for downstream use ──────────────────────────────────────────────
    all_site_results[reef_location] = {
        'metrics':           site_metrics,
        'labels':            labels,
        'masked_xgb':        masked_xgb,
        'masked_lin':        masked_lin,
        'masked_base':       masked_base,
        'mae_xgb_weekly':    mae_xgb_weekly,
        'mae_lin_weekly':    mae_lin_weekly,
        'mae_freq_weekly':   mae_freq_weekly,
        'shap_stacked':      all_shap_stacked,
        'expected_values':   all_fold_ev_arr,
        'renamed_features':  renamed_features,
        'X_values':          X_values,
        'n_years':           n_years,
        'site_name':         site_name,
    }

print(f"\n{'='*60}")
print(f'All sites complete: {list(all_site_results.keys())}')

## Per-Fold Performance (optional diagnostic)

Shows XGB and Linear metrics broken down by fold and year range.

In [ ]:
for reef_location in SITES_TO_RUN:
    res      = all_site_results[reef_location]
    labels   = res['labels']
    m_xgb    = res['masked_xgb']
    m_lin    = res['masked_lin']
    n_years  = res['n_years']
    site_name = res['site_name']

    splits_diag = create_splits(N_FOLDS, n_years, WEEKS_PER_YEAR)
    fold_size   = len(splits_diag[0][1])
    years_per_fold = fold_size // WEEKS_PER_YEAR

    fig, axes = plt.subplots(2, 3, figsize=(18, 5))
    axes = axes.flatten()
    model_colors = {'XGB': 'deeppink', 'Linear': 'gray'}
    metric_labels_diag = [
        f'Onset Acc\n(±{TOLERANCE}w)', 'No Onset\nAcc',
        'False Alarm\nRate', 'Missed\nOnset',
    ]
    start_year = 1985
    legend_handles = []

    for fold_idx, (_, test_idx) in enumerate(splits_diag):
        ax = axes[fold_idx]
        Y_fold = labels[test_idx]
        cutoff_f = np.repeat(VAL_NO_ONSET, len(Y_fold))
        preds_fold = {'XGB': m_xgb[test_idx], 'Linear': m_lin[test_idx]}

        fold_metrics = {}
        for mn, preds in preds_fold.items():
            hit  = (Y_fold < VAL_NO_ONSET) & (np.abs(preds - Y_fold) <= TOLERANCE)
            miss = (Y_fold < VAL_NO_ONSET) & (np.abs(preds - Y_fold) > TOLERANCE)
            fa   = (Y_fold == VAL_NO_ONSET) & (preds < cutoff_f)
            tno  = (Y_fold == VAL_NO_ONSET) & (preds >= cutoff_f)
            n_hit, n_miss, n_fa, n_tno = map(int, [np.sum(hit), np.sum(miss), np.sum(fa), np.sum(tno)])
            onset_acc = n_hit / (n_hit + n_miss) if (n_hit + n_miss) > 0 else 0
            no_onset  = n_tno / (n_tno + n_fa)   if (n_tno + n_fa)   > 0 else 0
            fa_rate   = n_fa  / (n_fa  + n_tno)  if (n_fa  + n_tno)  > 0 else 0
            miss_rate = n_miss/ (n_hit + n_miss)  if (n_hit + n_miss) > 0 else 0
            precision = n_hit / (n_hit + n_fa)    if (n_hit + n_fa)   > 0 else 0
            recall    = n_hit / (n_hit + n_miss)  if (n_hit + n_miss) > 0 else 0
            f1 = 2*precision*recall/(precision+recall) if (precision+recall)>0 else 0
            fold_metrics[mn] = [onset_acc*100, no_onset*100, fa_rate*100, miss_rate*100, f1]

        bar_width = 0.35
        x = np.arange(4)
        for i, (mn, color) in enumerate(model_colors.items()):
            bars = ax.bar(
                x + i*bar_width - bar_width/2, fold_metrics[mn][:4],
                width=bar_width, color=color, alpha=0.8,
                edgecolor='black' if mn == 'XGB' else 'none',
            )
            if fold_idx == 0:
                legend_handles.append(bars[0])

        Y_reshaped = Y_fold.reshape(years_per_fold, WEEKS_PER_YEAR)
        n_onsets = int(np.sum(np.any(Y_reshaped == 0, axis=1)))
        end_year = start_year + years_per_fold - 1
        ax.set_title(f'Fold {fold_idx+1} ({start_year}–{end_year}), Onsets: {n_onsets}', fontsize=14)
        ax.set_xticks(x)
        ax.set_xticklabels(metric_labels_diag, fontsize=11)
        ax.set_ylim(0, 100)
        ax.grid(axis='y', linestyle='--', alpha=0.5)
        ax.set_ylabel('Percentage (%)', fontsize=12)
        start_year += years_per_fold

    axes[-1].axis('off')
    axes[-1].legend(legend_handles, list(model_colors.keys()), fontsize=14, loc='center left')
    plt.suptitle(f'{site_name} — Per-Fold Performance', fontsize=16)
    plt.tight_layout()
    plt.show()